In [1]:
import pandas as pd

In [2]:
import pickle

In [3]:
import numpy as np

In [4]:
from sklearn.model_selection import train_test_split

In [5]:
from sklearn.preprocessing import StandardScaler

In [6]:
from sklearn.metrics import confusion_matrix, classification_report


In [7]:
from tensorflow.keras.models import Sequential

c:\Users\sanik\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\export\tf2onnx_lib.py:8: FutureWarning: In the future `np.object` will be defined as the corresponding NumPy scalar.
  if not hasattr(np, "object"):


In [8]:
from tensorflow.keras.layers import Dense

In [9]:
from tensorflow.keras.preprocessing.text import Tokenizer

In [10]:
from tensorflow.keras.preprocessing.sequence import pad_sequences

In [11]:
df = pd.read_csv("fake_account_detection_100k.csv")

In [12]:
df

,username,followers,following,posts,has_profile_pic,account_age_days,bio,fake
0,jkkfhzfug,113,1464,1,0,60,NaN,1
1,prgxhbnxrcye,195,1200,5,1,100,Official account,1
2,ubumroowr,23588,1705,139,1,3852,Tech enthusiast and blogger,0
3,bqsoyyxa,32299,474,65,1,1270,Tech enthusiast and blogger,0
4,kewuzoedmbrk,27,2037,4,0,51,Click link for giveaway,1
...,...,...,...,...,...,...,...,...
99995,nthac,30,1195,0,0,32,NaN,1
99996,hqkcrrwsmk,51,4275,2,1,85,NaN,1
99997,pwvcpuleb,27,2600,4,1,43,Follow me for free followers,1
99998,dgnyhnoz,7487,1503,466,1,897,Love traveling and photography,0


In [13]:
df.isnull().sum()

username               0
followers              0
following              0
posts                  0
has_profile_pic        0
account_age_days       0
bio                 9974
fake                   0
dtype: int64

In [14]:
df.duplicated().sum()

np.int64(0)

In [15]:
df["bio"] = df["bio"].fillna("")

In [16]:
df.isnull().sum()

username            0
followers           0
following           0
posts               0
has_profile_pic     0
account_age_days    0
bio                 0
fake                0
dtype: int64

In [17]:
tokenizer = Tokenizer(num_words=5000)
tokenizer.fit_on_texts(df["bio"])

In [18]:
bio_sequences = tokenizer.texts_to_sequences(df["bio"])
bio_padded = pad_sequences(bio_sequences, maxlen=20)

In [19]:
numeric_features = df[
    ["followers", "following", "posts", "has_profile_pic", "account_age_days"]
].values

In [20]:
scaler = StandardScaler()
numeric_features = scaler.fit_transform(numeric_features)

In [21]:
X = np.hstack((numeric_features, bio_padded))
y = df["fake"].values

In [22]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)


In [23]:
model = Sequential([
    Dense(64, activation="relu", input_shape=(X_train.shape[1],)),
    Dense(32, activation="relu"),
    Dense(1, activation="sigmoid")
])


c:\Users\sanik\AppData\Local\Programs\Python\Python311\Lib\site-packages\keras\src\layers\core\dense.py:106: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [24]:
model.compile(
    optimizer="adam",
    loss="binary_crossentropy",
    metrics=["accuracy"]
)

In [25]:
print("\nTraining Model...\n")
model.fit(X_train, y_train, epochs=20, batch_size=64)


Training Model...

Epoch 1/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9888 - loss: 0.0441
Epoch 2/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9995 - loss: 0.0019
Epoch 3/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 983us/step - accuracy: 0.9997 - loss: 9.4971e-04
Epoch 4/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 990us/step - accuracy: 0.9996 - loss: 0.0011
Epoch 5/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 926us/step - accuracy: 0.9998 - loss: 8.5866e-04
Epoch 6/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9997 - loss: 0.0013    
Epoch 7/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9999 - loss: 3.5748e-04
Epoch 8/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 994us/step - accuracy: 0.9999 - loss: 3.2212e-04
Epoch 9/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 0.9998 - loss: 6.0343e-04
Epoch 10/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accuracy: 0.9998 - loss: 9.2373e-04
Epoch 11/20
1250/1250 ━━━━━━━━━━━━━━━━━━━━ 2s 1ms/step - accura

In [26]:
loss, accuracy = model.evaluate(X_test, y_test)
print("\nTest Accuracy:", accuracy)

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 1ms/step - accuracy: 1.0000 - loss: 4.2767e-05

Test Accuracy: 1.0


In [27]:
y_pred = (model.predict(X_test) > 0.5).astype("int32")

625/625 ━━━━━━━━━━━━━━━━━━━━ 1s 890us/step


In [28]:
print("\nConfusion Matrix:")
print(confusion_matrix(y_test, y_pred))


Confusion Matrix:
[[10038     0]
 [    0  9962]]


In [29]:
print("\nClassification Report:")
print(classification_report(y_test, y_pred))


Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00     10038
           1       1.00      1.00      1.00      9962

    accuracy                           1.00     20000
   macro avg       1.00      1.00      1.00     20000
weighted avg       1.00      1.00      1.00     20000



In [30]:
model.save("fake_account_detection_model.h5")
print("\nModel saved as fake_account_detection_model.h5")


Model saved as fake_account_detection_model.h5


In [31]:
with open("tokenizer.pkl", "wb") as f:
    pickle.dump(tokenizer, f)

In [32]:
with open("scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)

In [33]:
print("Model, tokenizer, and scaler saved successfully!")

Model, tokenizer, and scaler saved successfully!


In [34]:
sample_account = {
    "followers": 50,
    "following": 3000,
    "posts": 2,
    "has_profile_pic": 0,
    "account_age_days": 20,
    "bio": "Follow me for free followers"
}

In [35]:
sample_bio_seq = tokenizer.texts_to_sequences([sample_account["bio"]])
sample_bio_pad = pad_sequences(sample_bio_seq, maxlen=20)

In [36]:
sample_numeric = np.array([[
    sample_account["followers"],
    sample_account["following"],
    sample_account["posts"],
    sample_account["has_profile_pic"],
    sample_account["account_age_days"]
]])

In [37]:
sample_numeric = scaler.transform(sample_numeric)


In [38]:
sample_input = np.hstack((sample_numeric, sample_bio_pad))

In [39]:
prediction = model.predict(sample_input)

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 92ms/step


In [40]:
print("\nPrediction for new account:")
if prediction[0][0] > 0.5:
    print("❌ Fake Account")
else:
    print("✅ Real Account")


Prediction for new account:
❌ Fake Account
